# Part 1 - Build the governed dataset

In [1]:
import re
import pandas as pd, numpy as np
from pathlib import Path

# Folder containing the capstone CSV files. Change this only if your CSVs are elsewhere.
DATA = Path(".")


def data_file(name):
    path = DATA / name
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path.resolve()}\n"
            "Put the capstone CSV files in DATA or change DATA above."
        )
    return path


s = pd.read_csv(data_file("capstone_subscribers.csv"))

print("rows in file          :", len(s))
print("distinct customer_id  :", s.customer_id.nunique())
print("duplicate customer_id :", s.duplicated(subset=["customer_id"]).sum())
print("distinct region values:", s.region.nunique())
print("missing avg_monthly_gb:", s.avg_monthly_gb.isna().sum())
print("missing days_since_last_recharge:", s.days_since_last_recharge.isna().sum())

rows in file          : 3222
distinct customer_id  : 3200
duplicate customer_id : 22
distinct region values: 18
missing avg_monthly_gb: 99
missing days_since_last_recharge: 58


In [2]:
# 1) text standardisation - do this BEFORE any groupby
s["region"] = s["region"].astype("string").str.strip().str.title()
print(
    "region values after cleaning:",
    s.region.nunique(),
    "->",
    sorted(s.region.dropna().unique()),
    s.region.nunique(),
    "->",
    sorted(s.region.dropna().unique()),
)

# 2) de-duplicate on the business key, keeping the first occurrence
before = len(s)
s = s.drop_duplicates(subset=["customer_id"], keep="first").copy()
print(f"removed {before - len(s)} duplicate subscriber rows, {len(s)} remain")

# 3) fill numeric gaps with a median and RECORD that you did it
for c in ["avg_monthly_gb", "days_since_last_recharge"]:
    s[c + "_was_missing"] = s[c].isna().astype(int)
    s[c] = s[c].fillna(s[c].median())
print(
    "rows flagged as imputed:",
    s[["avg_monthly_gb_was_missing", "days_since_last_recharge_was_missing"]]
    .sum()
    .to_dict(),
)

region values after cleaning: 6 -> ['Bengaluru', 'Chennai', 'Delhi', 'Hyderabad', 'Kolkata', 'Mumbai'] 6 -> ['Bengaluru', 'Chennai', 'Delhi', 'Hyderabad', 'Kolkata', 'Mumbai']
removed 22 duplicate subscriber rows, 3200 remain
rows flagged as imputed: {'avg_monthly_gb_was_missing': 96, 'days_since_last_recharge_was_missing': 58}


In [3]:
OUT_PATH = DATA / "capstone_integrated.csv"


def run_pipeline():
    df = pd.read_csv(data_file("capstone_subscribers.csv"))
    df["region"] = df["region"].astype("string").str.strip().str.title()
    df = df.drop_duplicates(subset=["customer_id"], keep="first").copy()

    # Apply the same missing-value repair used above and retain audit flags.
    for c in ["avg_monthly_gb", "days_since_last_recharge"]:
        df[c + "_was_missing"] = df[c].isna().astype(int)
        df[c] = df[c].fillna(df[c].median())

    # Keep this pipeline deterministic and safe to run repeatedly.
    df.to_csv(OUT_PATH, index=False)
    return len(df)


first = run_pipeline()
second = run_pipeline()
print(f"first run: {first} rows   second run: {second} rows")
assert first == second, f"pipeline is not idempotent: {first} -> {second}"
print("pipeline is idempotent")

first run: 3200 rows   second run: 3200 rows
pipeline is idempotent


# Part 2 - Track A - Churn prediction and retention strategy 

In [4]:
u = pd.read_csv(data_file("capstone_usage_monthly.csv"))

print(
    "rows in file:",
    len(u),
    "duplicate (customer_id, month):",
    u.duplicated(subset=["customer_id", "month"]).sum(),
)

# Remove duplicate customer-month records
u = u.drop_duplicates(subset=["customer_id", "month"], keep="first").copy()

u["month"] = u["month"].astype(str)

print("after de-duplication:", len(u), "rows across", u["month"].nunique(), "months")

# Aggregate usage at customer level
agg = (
    u.groupby("customer_id")
    .agg(
        months_seen=("month", "nunique"),
        tot_gb=("data_gb", "sum"),
        tot_voice=("voice_min", "sum"),
        tot_intl=("intl_min", "sum"),
        tot_rev=("revenue_inr", "sum"),
        zero_rev_months=("revenue_inr", lambda x: int((x == 0).sum())),
        fails=("failed_payment_count", "sum"),
    )
    .reset_index()
)

# Recent three months vs. earlier observations
late = (
    u[u["month"] >= "2026-05"]
    .groupby("customer_id")["data_gb"]
    .mean()
    .rename("gb_late")
)

early = (
    u[u["month"] < "2026-05"]
    .groupby("customer_id")["data_gb"]
    .mean()
    .rename("gb_early")
)

# Merge recent and early usage
agg = agg.merge(late, on="customer_id", how="left").merge(
    early, on="customer_id", how="left"
)

# Calculate usage trend
agg["gb_trend"] = agg["gb_late"] / agg["gb_early"].clip(lower=0.01)

# Merge aggregated features with the main customer dataframe
df = s.merge(agg, on="customer_id", how="left")

# Calculate complaint intensity
df["complaint_intensity"] = df["complaints_6m"] / (df["tenure_months"] / 6).clip(
    lower=1
)

print("merged frame:", df.shape)

rows in file: 19238 duplicate (customer_id, month): 38
after de-duplication: 19200 rows across 6 months
merged frame: (3200, 34)


In [5]:
# evidence 1 - the offer column is a consequence, not a cause
print(df.groupby("churn").retention_offer_sent.mean().round(3))

# evidence 2 - total_charges is arpu multiplied by tenure
implied = df.total_charges / df.tenure_months.clip(lower=1)
print("correlation of implied ARPU with arpu:", round(implied.corr(df.arpu), 4))

churn
0    0.051
1    0.843
Name: retention_offer_sent, dtype: float64
correlation of implied ARPU with arpu: 0.9995


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

LEAKY = ["retention_offer_sent", "total_charges"]


def make_preprocessor(X):
    num = X.select_dtypes(include=np.number).columns.tolist()
    cat = X.select_dtypes(exclude=np.number).columns.tolist()
    return ColumnTransformer(
        [
            (
                "n",
                Pipeline(
                    [("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]
                ),
                num,
            ),
            (
                "c",
                Pipeline(
                    [
                        ("i", SimpleImputer(strategy="most_frequent")),
                        ("o", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                cat,
            ),
        ],
        remainder="drop",
    )


def build_and_score(drop_cols, model=None):
    X = df.drop(columns=["churn", "customer_id"] + drop_cols, errors="ignore")
    y = df["churn"]
    pre = make_preprocessor(X)
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=42
    )
    clf = (
        model
        if model is not None
        else RandomForestClassifier(
            n_estimators=300, min_samples_leaf=3, random_state=42
        )
    )
    pipe = Pipeline([("pre", pre), ("clf", clf)]).fit(Xtr, ytr)
    proba = pipe.predict_proba(Xte)[:, 1]
    return roc_auc_score(yte, proba), proba, yte


# Show the effect of the two known leakage columns, then use the clean version.
auc_leak, _, _ = build_and_score([])
auc_clean, proba, yte = build_and_score(LEAKY)
print(f"ROC-AUC with the leak : {auc_leak:.3f}")
print(f"ROC-AUC without it    : {auc_clean:.3f}")
print(f"the leak was worth    : {auc_leak - auc_clean:.3f}")


ROC-AUC with the leak : 0.948
ROC-AUC without it    : 0.725
the leak was worth    : 0.223


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, accuracy_score

models = {
    "Logistic regression": LogisticRegression(max_iter=2000),
    "Decision tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "Random forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=3, random_state=42
    ),
}
rows = []
for name, m in models.items():
    a, pr, yt = build_and_score(LEAKY, m)
    p05 = (pr >= 0.5).astype(int)
    rows.append(
        {
            "model": name,
            "roc_auc": round(a, 3),
            "precision": round(precision_score(yt, p05, zero_division=0), 3),
            "recall": round(recall_score(yt, p05), 3),
            "accuracy": round(accuracy_score(yt, p05), 3),
        }
    )
pd.DataFrame(rows)

,model,roc_auc,precision,recall,accuracy
0,Logistic regression,0.741,0.708,0.240,0.794
1,Decision tree,0.649,0.500,0.188,0.760
2,Random forest,0.725,0.684,0.068,0.769


In [8]:
baseline = 1 - yte.mean()
print("predict nobody churns - accuracy :", round(baseline, 3))
print(
    "random forest at 0.5  - accuracy :",
    round(((proba >= 0.5).astype(int) == yte).mean(), 3),
)

predict nobody churns - accuracy : 0.76
random forest at 0.5  - accuracy : 0.769


In [9]:
from sklearn.metrics import confusion_matrix

COST_FN, COST_FP = 5880, 300

rows = []
for t in np.arange(0.05, 0.95, 0.05):
    pred = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
    rows.append(
        {
            "threshold": round(t, 2),
            "precision": round(precision_score(yte, pred, zero_division=0), 3),
            "recall": round(recall_score(yte, pred), 3),
            "flagged": int(pred.sum()),
            "expected_cost": fn * COST_FN + fp * COST_FP,
        }
    )
sweep = pd.DataFrame(rows)
sweep.sort_values("expected_cost").head(4)

,threshold,precision,recall,flagged,expected_cost
1,0.10,0.256,0.995,747,172680
0,0.05,0.240,0.995,796,187380
2,0.15,0.279,0.932,642,215340
3,0.20,0.316,0.812,494,313080


In [10]:
# the closed-form optimum, for comparison with your sweep
print("theoretical optimum:", round(COST_FP / (COST_FP + COST_FN), 4))

theoretical optimum: 0.0485


In [11]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

feats = [
    "avg_monthly_gb",
    "avg_voice_min",
    "tenure_months",
    "arpu",
    "complaints_6m",
    "days_since_last_recharge",
]
X = SimpleImputer(strategy="median").fit_transform(df[feats])
X = StandardScaler().fit_transform(X)  # never skip this

for kk in range(2, 9):
    km = KMeans(n_clusters=kk, n_init=10, random_state=42).fit(X)
    sil = silhouette_score(X, km.labels_, sample_size=2000, random_state=1)
    print(f"k={kk}  inertia={km.inertia_:9.1f}  silhouette={sil:.3f}")

k=2  inertia=  15575.2  silhouette=0.238
k=3  inertia=  13853.9  silhouette=0.235
k=4  inertia=  12290.8  silhouette=0.204
k=5  inertia=  10913.5  silhouette=0.212
k=6  inertia=   9710.8  silhouette=0.205
k=7  inertia=   9044.2  silhouette=0.176
k=8  inertia=   8674.9  silhouette=0.158


In [12]:
K = 4  # your choice - change it and justify it
km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(X)
df["segment"] = km.labels_

profile = (
    df.groupby("segment")
    .agg(
        size=("customer_id", "count"),
        mean_arpu=("arpu", "mean"),
        mean_tenure=("tenure_months", "mean"),
        mean_gb=("avg_monthly_gb", "mean"),
        mean_complaints=("complaints_6m", "mean"),
        churn_rate=("churn", "mean"),
    )
    .round(2)
)
profile["share_pct"] = (profile["size"] / len(df) * 100).round(1)
profile

,size,mean_arpu,mean_tenure,mean_gb,mean_complaints,churn_rate,share_pct
segment,,,,,,,
0,392,239.49,26.54,7.37,3.37,0.40,12.2
1,1628,204.37,21.68,5.74,0.46,0.23,50.9
2,661,406.68,24.02,18.66,0.69,0.21,20.7
3,519,249.96,25.47,6.59,0.57,0.19,16.2


In [13]:
from sklearn.ensemble import IsolationForest

u["intl_share"] = u.intl_min / (u.voice_min + u.intl_min + 1)
u["revenue_per_gb"] = u.revenue_inr / u.data_gb.clip(lower=0.01)
u["zero_revenue"] = (u.revenue_inr == 0).astype(int)
u["data_to_voice"] = u.data_gb / u.voice_min.clip(lower=1)

feat = [
    "data_gb",
    "voice_min",
    "intl_min",
    "sms_count",
    "revenue_inr",
    "failed_payment_count",
    "intl_share",
    "revenue_per_gb",
    "data_to_voice",
]
Xa = StandardScaler().fit_transform(u[feat].replace([np.inf, -np.inf], 0).fillna(0))

for c in [0.005, 0.01, 0.02]:
    iso = IsolationForest(contamination=c, n_estimators=300, random_state=42).fit(Xa)
    u[f"flag_{c}"] = (iso.predict(Xa) == -1).astype(int)
    u[f"score_{c}"] = -iso.score_samples(Xa)
    n = int(u[f"flag_{c}"].sum())
    print(
        f"contamination {c:<6} flagged {n:>4} rows"
        f"  ~{n / 26:.1f}/week   cost Rs {n * 450:,}"
    )

contamination 0.005  flagged   96 rows  ~3.7/week   cost Rs 43,200
contamination 0.01   flagged  192 rows  ~7.4/week   cost Rs 86,400
contamination 0.02   flagged  384 rows  ~14.8/week   cost Rs 172,800


In [14]:
# Score every subscriber, not just the held-out fold.
# Rebuild the preprocessor from the clean feature set so this cell does not
# depend on a local variable created inside build_and_score().
X_all = df.drop(columns=["churn", "customer_id"] + LEAKY, errors="ignore")
pre_all = make_preprocessor(X_all)
full = Pipeline(
    [
        ("pre", pre_all),
        (
            "clf",
            RandomForestClassifier(
                n_estimators=300, min_samples_leaf=3, random_state=42
            ),
        ),
    ]
)
full.fit(X_all, df["churn"])
df["risk"] = full.predict_proba(X_all)[:, 1]

df["expected_loss"] = df.risk * df.arpu * 12  # a year of revenue at risk
call_list = df.sort_values("expected_loss", ascending=False)
print(call_list[["customer_id", "segment", "risk", "arpu", "expected_loss"]].head(10))

# Compare risk-only targeting with value-at-risk targeting.
top_n = min(300, len(df))
top_risk = set(df.nlargest(top_n, "risk").customer_id)
top_value = set(df.nlargest(top_n, "expected_loss").customer_id)
print("overlap between the two top lists:", len(top_risk & top_value))


     customer_id  segment      risk    arpu  expected_loss
2747   SUB101910        2  0.640496  660.72    5078.264657
1235   SUB101691        3  0.590526  528.97    3748.446563
1538   SUB102827        2  0.568370  535.53    3652.549556
1103   SUB100523        0  0.638075  471.02    3606.552596
3153   SUB101952        0  0.673428  435.45    3518.930060
1338   SUB100580        2  0.577793  504.33    3496.782062
693    SUB102863        2  0.606318  475.08    3456.594331
1815   SUB102265        2  0.554915  518.01    3449.421305
643    SUB100865        2  0.615527  462.15    3413.591684
1406   SUB101169        2  0.579672  480.89    3345.104335
overlap between the two top lists: 131


In [15]:
kp = pd.read_csv(data_file("capstone_network_kpi.csv"))
print("rows in file             :", len(kp))
print("distinct cells           :", kp.cell_id.nunique())
print(
    "duplicate (cell_id, ts)  :", kp.duplicated(subset=["cell_id", "reading_ts"]).sum()
)
print("blank latency_ms         :", kp.latency_ms.isna().sum())
print("negative throughput_mbps :", (kp.throughput_mbps < 0).sum())
print("rows with downtime > 0   :", (kp.downtime_min > 0).sum())


rows in file             : 8681
distinct cells           : 18
duplicate (cell_id, ts)  : 41
blank latency_ms         : 97
negative throughput_mbps : 9
rows with downtime > 0   : 34


In [16]:
kc = kp.drop_duplicates(subset=["cell_id", "reading_ts"], keep="first").copy()
kc["ts"] = pd.to_datetime(kc["reading_ts"], errors="coerce")
if kc["ts"].isna().any():
    raise ValueError("Some reading_ts values could not be parsed as timestamps.")
print("after de-duplication:", len(kc), "rows")

mw = pd.read_csv(data_file("capstone_maintenance_windows.csv"))
mw["window_start"] = pd.to_datetime(mw["window_start"], errors="coerce")
mw["window_end"] = pd.to_datetime(mw["window_end"], errors="coerce")

in_maint = pd.Series(False, index=kc.index)
for _, w in mw.iterrows():
    in_maint |= (
        (kc["cell_id"] == w["cell_id"])
        & (kc["ts"] >= w["window_start"])
        & (kc["ts"] < w["window_end"])
    )

blank = kc["latency_ms"].isna()
outage = blank & ~in_maint & (kc["downtime_min"] > 0)
print("blank total                        :", int(blank.sum()))
print("blank inside a maintenance window  :", int((blank & in_maint).sum()))
print("blank during an unplanned outage   :", int(outage.sum()))
print("blank with no explanation at all   :", int((blank & ~in_maint & ~outage).sum()))


after de-duplication: 8640 rows
blank total                        : 90
blank inside a maintenance window  : 20
blank during an unplanned outage   : 7
blank with no explanation at all   : 63


In [17]:
# negative throughput is a counter rollover, not a measurement of anything
neg = kc.throughput_mbps < 0
kc.loc[neg, "throughput_mbps"] = None
print("negative throughput nulled:", int(neg.sum()))
print("blanks left untouched     :", int(kc.latency_ms.isna().sum()))

negative throughput nulled: 9
blanks left untouched     : 90


In [18]:
import sqlite3

con = sqlite3.connect("capstone_rt.db")
con.executescript(""" 
CREATE TABLE IF NOT EXISTS kpi_readings ( 
  reading_ts        TEXT NOT NULL, 
  cell_id           TEXT NOT NULL, 
  region            TEXT NOT NULL, 
  latency_ms        REAL, 
  throughput_mbps   REAL, 
  downtime_min      INTEGER NOT NULL DEFAULT 0, 
  drop_call_rate_pct REAL, 
  active_users      INTEGER, 
  UNIQUE (cell_id, reading_ts) 
); 
CREATE INDEX IF NOT EXISTS ix_cell_ts ON kpi_readings (cell_id, reading_ts); 
""")
con.commit()

cols = [
    "reading_ts",
    "cell_id",
    "region",
    "latency_ms",
    "throughput_mbps",
    "downtime_min",
    "drop_call_rate_pct",
    "active_users",
]
recs = kc[cols].astype(object).where(pd.notna(kc[cols]), None).values.tolist()


def load(rows):
    con.executemany(
        "INSERT OR IGNORE INTO kpi_readings "
        "(reading_ts, cell_id, region, latency_ms, throughput_mbps, "
        "downtime_min, drop_call_rate_pct, active_users) "
        "VALUES (?,?,?,?,?,?,?,?)",
        rows,
    )
    con.commit()


load(recs)
print(
    "after first load :", con.execute("SELECT COUNT(*) FROM kpi_readings").fetchone()[0]
)
load(recs)
print(
    "after second load:", con.execute("SELECT COUNT(*) FROM kpi_readings").fetchone()[0]
)

after first load : 8640
after second load: 8640


In [19]:
q = """SELECT reading_ts, latency_ms FROM kpi_readings 
       WHERE cell_id = 'CELL-DEL-03' AND reading_ts LIKE '2026-08-03%' 
       ORDER BY latency_ms DESC LIMIT 3"""
for r in con.execute(q):
    print(r)

('2026-08-03 20:00:00', 117.25)
('2026-08-03 20:45:00', 117.08)
('2026-08-03 21:30:00', 116.37)


In [20]:
q = """SELECT substr(reading_ts,1,13) || ':00' AS window_start, 
              ROUND(AVG(latency_ms),2) AS avg_latency, 
              ROUND(MAX(latency_ms),2) AS peak_latency, 
              COUNT(*) AS readings 
       FROM kpi_readings 
       WHERE cell_id = 'CELL-DEL-03' AND reading_ts LIKE '2026-08-03%' 
       GROUP BY window_start ORDER BY avg_latency DESC LIMIT 3"""
for r in con.execute(q):
    print(r)

('2026-08-03 21:00', 110.72, 116.37, 4)
('2026-08-03 20:00', 107.94, 117.25, 4)
('2026-08-03 19:00', 82.48, 89.86, 4)


In [21]:
peak = con.execute("""SELECT MAX(latency_ms) FROM kpi_readings 
   WHERE cell_id='CELL-DEL-03' AND reading_ts LIKE '2026-08-03%'""").fetchone()[0]

four_h = con.execute("""SELECT MAX(a) FROM ( 
   SELECT AVG(latency_ms) a FROM kpi_readings 
   WHERE cell_id='CELL-DEL-03' AND reading_ts LIKE '2026-08-03%' 
   GROUP BY CAST(substr(reading_ts,12,2) AS INTEGER)/4)""").fetchone()[0]

raw = con.execute("""SELECT COUNT(*) FROM kpi_readings 
   WHERE cell_id='CELL-DEL-03' AND reading_ts LIKE '2026-08-03%' 
   AND latency_ms > 100""").fetchone()[0]

print(f"true 15-minute peak                : {peak:.2f} ms")
print(f"highest 4-hour mean                : {four_h:.2f} ms")
print(f"understated by                     : {(peak - four_h) / peak * 100:.1f}%")
print(f"breaches of 100 ms on the raw grain: {raw}")
print(f"breaches of 100 ms on the aggregate: {int(four_h > 100)}")

true 15-minute peak                : 117.25 ms
highest 4-hour mean                : 76.22 ms
understated by                     : 35.0%
breaches of 100 ms on the raw grain: 8
breaches of 100 ms on the aggregate: 0


In [22]:
THRESHOLD = 70  # ms

# stage 0 - the naive rule on the raw file
raw_alerts = ((kp.latency_ms > THRESHOLD) | (kp.downtime_min > 0)).sum()

# stage 1 - the same rule on the de-duplicated table
dedup_alerts = ((kc.latency_ms > THRESHOLD) | (kc.downtime_min > 0)).sum()

# stage 2 - drop anything inside a maintenance window
supp = ((kc.latency_ms > THRESHOLD) | (kc.downtime_min > 0)) & ~in_maint

# stage 3 - require three consecutive intervals (45 minutes)
fire = kc[supp].sort_values(["cell_id", "ts"])
runs = []
for cid, g in fire.groupby("cell_id"):
    t = g["ts"].tolist()
    if not t:
        continue
    start = prev = t[0]
    n = 1
    for x in t[1:]:
        if (x - prev) == pd.Timedelta(minutes=15):
            n += 1
        else:
            runs.append((cid, start, n))
            start = x
            n = 1
        prev = x
    runs.append((cid, start, n))
long_runs = [r for r in runs if r[2] >= 3]

print("raw rule on the raw file  :", int(raw_alerts))
print("after de-duplication      :", int(dedup_alerts))
print("after maintenance suppression:", int(supp.sum()))
print("after 3-interval persistence :", sum(r[2] for r in long_runs))
print("after hysteresis grouping    :", len(long_runs))
for cid, start, n in sorted(long_runs, key=lambda r: -r[2]):
    print(f"   {cid}  from {start}  ({n} intervals)")

raw rule on the raw file  : 75
after de-duplication      : 63
after maintenance suppression: 43
after 3-interval persistence : 27
after hysteresis grouping    : 3
   CELL-DEL-03  from 2026-08-03 19:00:00  (13 intervals)
   CELL-CHE-02  from 2026-08-04 19:15:00  (7 intervals)
   CELL-HYD-01  from 2026-08-02 09:15:00  (7 intervals)


In [23]:
t = pd.read_csv(data_file("capstone_cell_traffic_monthly.csv"))

t["Month"] = pd.PeriodIndex(t["Month"].astype(str), freq="M").to_timestamp()

t = t.sort_values(["Cell", "Month"]).copy()

# Fill missing traffic values within each cell
t["Traffic_GB"] = t.groupby("Cell")["Traffic_GB"].transform(
    lambda x: x.interpolate().bfill().ffill()
)

# Log-space forecasting requires strictly positive traffic
t["Traffic_GB"] = t["Traffic_GB"].clip(lower=0.01)


def forecast_cell(g, horizon=6):
    g = g.sort_values("Month").reset_index(drop=True)

    # Convert traffic to log-space
    y = np.log(g["Traffic_GB"].to_numpy())

    # Time index
    x = np.arange(len(y))

    # Fit linear trend in log-space
    slope, intercept = np.polyfit(x, y, 1)

    # Calculate residuals
    resid = y - (intercept + slope * x)

    # Estimate monthly seasonal effect
    seas = pd.Series(resid).groupby(g["Month"].dt.month.to_numpy()).mean()

    # Future time points
    fx = np.arange(len(y), len(y) + horizon)

    # Future months
    fm = pd.date_range(
        g["Month"].iloc[-1] + pd.offsets.MonthBegin(1), periods=horizon, freq="MS"
    )

    # Forecast in log-space
    yhat = intercept + slope * fx + np.array([seas.get(m.month, 0.0) for m in fm])

    # Convert back from log-space
    return pd.DataFrame(
        {
            "Cell": g["Cell"].iloc[0],
            "Month": fm,
            "Forecast_GB": np.round(np.exp(yhat), 1),
            "Capacity_GB": g["Capacity_GB"].iloc[-1],
        }
    )


# Generate forecasts for every cell
fc = pd.concat([forecast_cell(g) for _, g in t.groupby("Cell")], ignore_index=True)

# Calculate forecast utilisation percentage
fc["util_pct"] = (fc["Forecast_GB"] / fc["Capacity_GB"] * 100).round(1)


# Get the latest traffic record for each cell
today = t.groupby("Cell").tail(1)

# Calculate current utilisation
util_now = (
    today.set_index("Cell")["Traffic_GB"] / today.set_index("Cell")["Capacity_GB"] * 100
).round(1)

print("cells above 85% utilisation today:", int((util_now > 85).sum()))

print(util_now.nlargest(5).to_string())

cells above 85% utilisation today: 3
Cell
CELL-KOL-03    87.4
CELL-MUM-01    87.4
CELL-HYD-03    87.1
CELL-DEL-02    84.1
CELL-BLR-01    81.9


In [24]:
exposure = (
    s.groupby("home_cell_id")
    .agg(subs=("customer_id", "count"), mean_arpu=("arpu", "mean"))
    .assign(monthly_revenue=lambda d: d.subs * d.mean_arpu)
    .reset_index()
    .rename(columns={"home_cell_id": "cell_id"})
)

quality = (
    kc.groupby("cell_id")
    .agg(
        p95_latency=("latency_ms", lambda x: x.quantile(0.95)),
        downtime_intervals=("downtime_min", lambda x: (x > 0).sum()),
    )
    .reset_index()
)

risk = quality.merge(exposure, on="cell_id")
risk["revenue_at_risk"] = risk.monthly_revenue * (risk.p95_latency > 70)
print(
    risk.sort_values("revenue_at_risk", ascending=False).head(8).to_string(index=False)
)

    cell_id  p95_latency  downtime_intervals  subs  mean_arpu  monthly_revenue  revenue_at_risk
CELL-BLR-01      54.0865                   0   177 247.474972         43803.07              0.0
CELL-BLR-02      41.3800                   6   191 265.593455         50728.35              0.0
CELL-BLR-03      59.8920                   0   176 258.775625         45544.51              0.0
CELL-CHE-01      46.2225                   0   172 264.365174         45470.81              0.0
CELL-CHE-02      42.6615                   0   175 257.538686         45069.27              0.0
CELL-CHE-03      51.5560                   0   178 258.539382         46020.01              0.0
CELL-DEL-01      42.7710                   0   181 265.747956         48100.38              0.0
CELL-DEL-02      58.6060                   0   186 261.248495         48592.22              0.0


In [25]:
m = pd.read_csv(data_file("capstone_messages.csv"))
print("messages:", len(m))
print(m.true_sentiment.value_counts().to_string())

j = m.merge(
    s[["customer_id", "arpu", "tenure_months", "region", "plan_type"]],
    on="customer_id",
    how="left",
)
unmatched = j.arpu.isna().sum()
print("messages with no matching subscriber:", unmatched)


messages: 354
true_sentiment
Negative    152
Positive    107
Neutral      95
messages with no matching subscriber: 18


In [26]:
import re
import pandas as pd
from sklearn.metrics import accuracy_score

POSITIVE = {
    "amazing",
    "happy",
    "great",
    "love",
    "easy",
    "quick",
    "excellent",
    "satisfied",
    "best",
    "thank",
    "well",
    "correct",
    "pleased",
    "fantastic",
    "good",
    "fair",
    "fast",
    "minutes",
    "solved",
}

NEGATIVE = {
    "terrible",
    "drop",
    "drops",
    "overcharged",
    "unacceptable",
    "no",
    "awful",
    "worst",
    "waited",
    "nothing",
    "disappointed",
    "frustrated",
    "failed",
    "slow",
    "expensive",
    "porting",
    "unresolved",
    "not",
}


def score_message(text):
    # Remove punctuation and convert text to lowercase
    words = re.findall(r"[a-z']+", str(text).lower())

    # Count positive and negative words
    pos = sum(word in POSITIVE for word in words)
    neg = sum(word in NEGATIVE for word in words)

    if pos > neg:
        return "Positive", pos - neg

    elif neg > pos:
        return "Negative", pos - neg

    else:
        return "Neutral", 0


# Apply sentiment scoring
m[["predicted", "score"]] = m["message_text"].apply(
    lambda text: pd.Series(score_message(text))
)


# Calculate overall accuracy
print(
    "accuracy, all messages :",
    round(accuracy_score(m["true_sentiment"], m["predicted"]) * 100, 1),
    "%",
)


# Calculate accuracy for each case type
for grp, g in m.groupby("case_type"):
    print(
        f"accuracy, {grp:9} cases: "
        f"{accuracy_score(g['true_sentiment'], g['predicted']) * 100:5.1f}% "
        f"(n={len(g)})"
    )

accuracy, all messages : 92.1 %
accuracy, hard      cases:  25.0% (n=24)
accuracy, standard  cases:  97.0% (n=330)


In [27]:
# OPTIONAL - requires transformers + torch, pre-installed and cached.
# If this cell fails, note it in the notebook and move on to C4.
from transformers import pipeline

sentiment_ai = pipeline(
    "sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english"
)

hard = m[m.case_type == "hard"].drop_duplicates("message_text")
for _, r in hard.iterrows():
    ours, _ = score_message(r.message_text)
    ai = sentiment_ai(r.message_text)[0]["label"].title()
    print(
        f"{r.message_text[:46]:48} true={r.true_sentiment:9} ours={ours:9} model={ai}"
    )

Device set to use cpu


Great. Another dropped call. Exactly what I ne   true=Negative  ours=Positive  model=Positive
I was worried about my bill but it turned out    true=Positive  ours=Neutral   model=Positive
Oh brilliant, no signal AGAIN, I just love pay   true=Negative  ours=Negative  model=Positive
The staff were not helpful at all                true=Negative  ours=Negative  model=Negative
Not the worst month I have had with you          true=Positive  ours=Negative  model=Positive
Nothing about this service is good               true=Negative  ours=Neutral   model=Negative
The network is not bad actually                  true=Positive  ours=Negative  model=Positive
I would not say I am unhappy with the speed      true=Positive  ours=Negative  model=Positive


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# This section trains a simple ticket router.  Use the same train/test split
# for the vectorizer to avoid test-set vocabulary leakage.
tk = pd.read_csv(data_file("capstone_tickets.csv"))
print("tickets:", len(tk), " teams:", tk.team.nunique())

X_text = tk.ticket_text.fillna("").astype(str)
y = tk.team
Xtr_text, Xte_text, ytr, yte = train_test_split(
    X_text, y, test_size=0.25, random_state=42, stratify=y
)
vec = TfidfVectorizer()
Xtr = vec.fit_transform(Xtr_text)
Xte = vec.transform(Xte_text)
print("each ticket is now", Xtr.shape[1], "number-columns")

router = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
pred = router.predict(Xte)
print("routing accuracy:", round(accuracy_score(yte, pred) * 100, 1), "%")


tickets: 548  teams: 5
each ticket is now 195 number-columns
routing accuracy: 88.3 %


In [29]:
# Confidence is measured on the complete ticket set using a vectorizer
# fitted only on the training split above.
X_all_tickets = vec.transform(tk.ticket_text.fillna("").astype(str))
conf = router.predict_proba(X_all_tickets).max(axis=1)
for floor in [0.5, 0.6, 0.7, 0.8, 0.9]:
    share = (conf < floor).mean() * 100
    print(f"floor {floor:.0%}: {share:5.1f}% of all tickets go to a human")
print("median confidence:", round(float(np.median(conf)), 3))

amb = tk[tk.case_type == "ambiguous"]
ca = router.predict_proba(vec.transform(amb.ticket_text.fillna("").astype(str))).max(
    axis=1
)
print(
    "ambiguous tickets:",
    len(amb),
    " mean confidence:",
    round(float(ca.mean()), 3) if len(ca) else "n/a",
)

print(
    "ambiguous tickets:",
    len(amb),
    " mean confidence:",
    round(float(ca.mean()), 3) if len(ca) else "n/a",
)


floor 50%:   4.7% of all tickets go to a human
floor 60%:  14.8% of all tickets go to a human
floor 70%:  42.7% of all tickets go to a human
floor 80%:  86.1% of all tickets go to a human
floor 90%: 100.0% of all tickets go to a human
median confidence: 0.712
ambiguous tickets: 36  mean confidence: 0.625
ambiguous tickets: 36  mean confidence: 0.625


In [30]:
j["is_negative"] = (
    j.message_text.apply(lambda t: score_message(t)[0]) == "Negative"
).astype(int)
j["anger"] = -j.message_text.apply(lambda t: score_message(t)[1])
j["priority"] = j.anger * j.arpu.fillna(j.arpu.median())

queue = j.sort_values("priority", ascending=False)
print(
    queue[["message_id", "customer_id", "arpu", "anger", "priority", "message_text"]]
    .head(10)
    .to_string(index=False)
)
queue.head(50).to_csv("escalation_list.csv", index=False)


message_id customer_id   arpu  anger  priority                                                      message_text
  MSG60171   SUB102549 504.94      3   1514.82    Slow internet, expensive plan, a terrible experience all round
  MSG60013   SUB102060 483.31      3   1449.93 Worst service I have ever had, I waited two hours and got nothing
  MSG60176   SUB102318 407.45      3   1222.35    Slow internet, expensive plan, a terrible experience all round
  MSG60245   SUB101836 399.59      3   1198.77 Worst service I have ever had, I waited two hours and got nothing
  MSG60338   SUB102050 378.30      3   1134.90    Slow internet, expensive plan, a terrible experience all round
  MSG60156   SUB100528 352.82      3   1058.46 Worst service I have ever had, I waited two hours and got nothing
  MSG60026   SUB102779 352.64      3   1057.92    Slow internet, expensive plan, a terrible experience all round
  MSG60056   SUB101791 339.19      3   1017.57    Slow internet, expensive plan, a terrible expe

# Part 3 - Turn the analysis into a decision 


In [31]:
# Decision model: target the highest-value/highest-risk subscribers from the
# clean churn model above.  This replaces the unfinished ... placeholders.
target_n = min(300, len(df))
target_group = df.nlargest(target_n, "expected_loss")
segment_size = len(target_group)
base_churn = float(target_group["risk"].mean())
value_lost = 5880  # lifetime value of a lost subscriber

print(f"target subscribers: {segment_size}")
print(f"average predicted churn risk: {base_churn:.3f}")

options = [
    {"name": "Do nothing", "cost_per_sub": 0, "reduction": 0.00},
    {"name": "Retention call", "cost_per_sub": 300, "reduction": 0.15},
    {"name": "Bill credit + call", "cost_per_sub": 550, "reduction": 0.25},
]

for o in options:
    saved = segment_size * base_churn * o["reduction"]
    spend = segment_size * o["cost_per_sub"]
    o["net"] = saved * value_lost - spend
    print(
        f"{o['name']:22} saves {saved:6.0f} subs   "
        f"spend Rs {spend:>10,.0f}   net Rs {o['net']:>12,.0f}"
    )

best = max(options, key=lambda o: o["net"])
print(f"recommended option: {best['name']} (net Rs {best['net']:,.0f})")


target subscribers: 300
average predicted churn risk: 0.561
Do nothing             saves      0 subs   spend Rs          0   net Rs            0
Retention call         saves     25 subs   spend Rs     90,000   net Rs       58,326
Bill credit + call     saves     42 subs   spend Rs    165,000   net Rs       82,210
recommended option: Bill credit + call (net Rs 82,210)
